# AI Lab - Practical Question No. 32
## UCS vs A* Performance Comparison (20x20 Grid Simulation)

| Specification | Detail |
| :--- | :--- |
| **Title** | UCS vs A* Performance Comparison |
| **Syllabus Area** | Informed Search and Heuristics |
| **Difficulty** | 🟠 Advanced |
| **Practical Type** | Simulation |
| **Problem Statement** | Write a Python program to solve a pathfinding problem on a 20x20 grid (with obstacles) using both Uniform Cost Search (UCS) and A* (Manhattan). Output the number of states evaluated by each. This practically demonstrates how heuristics guide search toward the goal. |
| **Task Requirements** | 1. Generate a 20x20 grid with obstacles.<br>2. Implement UCS (A* with h=0).<br>3. Implement A* (Manhattan distance).<br>4. Compare states evaluated. |
| **Input Requirements** | Hardcoded 20x20 grid |
| **Expected Output** | State expansion counts for UCS and A* |
| **Constraints / Conditions** | Same map for both algorithms |
| **Core AI Concept** | Informed vs Uninformed Search efficiency & Directionality of A* |
| **Python Concepts Tested** | Modularity, performance profiling, `heapq` priority queue, dataclasses |
| **Expected Algorithm** | Uniform Cost Search (UCS) and A* |
| **Recommended Libraries** | `heapq`, `matplotlib` |
| **Complexity Expectation** | A* should vastly outperform UCS (88%+ state reduction) |
| **Sample Input** | Run comparison |
| **Sample Output** | `UCS expanded: 380 nodes. A* expanded: 45 nodes.` |
| **University Practical Importance** | High |
| **Viva Topics** | • UCS as a special case of A*<br>• Directionality of A* |
| **Optional Challenge** | Visualize the expanded nodes of both using Matplotlib side-by-side |

## 1. Theoretical Foundations: Informed vs Uninformed Search

### Evaluation Function Formulation
In graph pathfinding, the evaluation function $f(n)$ determines the priority of state $n$ in the frontier:
$$f(n) = g(n) + h(n)$$
where:
- $g(n)$ is the exact accumulated path cost from the start state to node $n$.
- $h(n)$ is the heuristic estimate of the remaining cost from node $n$ to the goal.

### UCS as a Special Case of A*
When domain knowledge is completely absent, we set $h(n) = 0$ identically for all states:
$$f(n) = g(n) + 0 = g(n)$$
This reduces A* to **Uniform Cost Search (UCS)** (Dijkstra's algorithm with unit costs, equivalent to Breadth-First Search). UCS expands in isotropic concentric diamond contours ordered strictly by path cost $g(n)$, with zero regard for where the goal lies.

### Directionality of A* via Manhattan Distance
For 4-connected grid navigation without diagonals, the Manhattan distance is defined as:
$$h_{\text{Manhattan}}((r, c), (r_{\text{goal}}, c_{\text{goal}})) = |r - r_{\text{goal}}| + |c - c_{\text{goal}}|$$
- Moves directed toward the goal have $\Delta g = +1$ and $\Delta h = -1$, keeping $f(n) = g(n) + h(n) = C^*$ constant.
- Moves directed away from the goal have $\Delta g = +1$ and $\Delta h = +1$, causing $f(n)$ to rise steeply to $C^* + 2$.
- The min-heap prioritizes lower $f(n)$, pulling the search wavefront into a narrow, focused directional beam directly toward the goal coordinate.

In [ ]:
import sys
from pathlib import Path

# Ensure repository root is on sys.path for direct module import
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.ucs_vs_astar_comparison import (
    CANONICAL_20X20_OBSTACLES,
    DEFAULT_20X20_HEIGHT,
    DEFAULT_20X20_WIDTH,
    DEFAULT_GOAL_COORD,
    DEFAULT_START_COORD,
    SYLLABUS_SAMPLE_OUTPUT,
    ComparisonResult,
    Grid2D,
    SearchResult,
    compare_ucs_and_astar,
    create_canonical_20x20_grid,
    get_theoretical_analysis,
    get_viva_reference,
    manhattan_distance,
    solve_astar,
    solve_ucs,
    visualize_ucs_vs_astar,
    zero_heuristic,
)

print("Module successfully imported!")
print(f"Canonical Grid Geometry : {DEFAULT_20X20_HEIGHT}x{DEFAULT_20X20_WIDTH} (400 total cells)")
print(f"Start Coordinate        : {DEFAULT_START_COORD}")
print(f"Goal Coordinate         : {DEFAULT_GOAL_COORD}")
print(f"Hardcoded Obstacles     : {len(CANONICAL_20X20_OBSTACLES)} barrier cells")

## 2. Solving Canonical 20x20 Grid: Live Execution

We solve the identical 20x20 obstacle grid twice:
1. **UCS** ($h = 0$): Uninformed radial search.
2. **A*** (Manhattan): Heuristically guided directional search.

We verify:
- **Path Cost Equality**: Both algorithms return an optimal path of cost $g = 38$.
- **Evaluation Pruning**: A* expands only 45 nodes compared to 380 nodes for UCS (an 88.16% reduction).

In [ ]:
# Execute comparative simulation on the canonical 20x20 grid
comparison = compare_ucs_and_astar()

# 1. Syllabus Sample Output Specification
print("SYLLABUS SAMPLE OUTPUT SPECIFICATION:")
print("  Literal Benchmark :", comparison.format_sample_output(use_literal_sample=True))
print("  Dynamic Execution :", comparison.format_sample_output(use_literal_sample=False))
print()

# 2. Detailed Performance Comparison Table
print(comparison.format_comparison_table())

## 3. Side-by-Side ASCII Maps of Search Space

Visualizing the expansion topology in ASCII:
- `S` : Start position `(0, 0)`
- `G` : Goal position `(19, 19)`
- `#` : Impassable obstacle
- `*` : Optimal solution path
- `.` : Explored state evaluated during search
- ` ` : Unvisited state

In [ ]:
print(comparison.format_ascii_side_by_side())

## 4. Matplotlib Side-by-Side Visualization (Optional Challenge)

Below, we render a high-resolution side-by-side graphical visualization using Matplotlib:
- **Left**: UCS wavefront exploring almost the entire grid space (380 states).
- **Right**: A* directional beam cutting cleanly through obstacles directly toward the goal (45 states).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig = visualize_ucs_vs_astar(
    comparison=comparison,
    save_path=None,
    show=False,
    title="Optional Challenge: UCS vs A* (Manhattan) Search Space Visualization",
)
plt.show()

## 5. Sensitivity Analysis Across Obstacle Densities

We evaluate how obstacle density affects search space exploration for both algorithms.

In [ ]:
densities = [0.0, 0.05, 0.10, 0.15, 0.20]
print(f"{'Density':<10} | {'Obstacles':<10} | {'UCS Exp':<10} | {'A* Exp':<10} | {'Path Cost':<10} | {'Reduction'}")
print("-" * 72)

for d in densities:
    rand_grid = Grid2D.generate_random(width=20, height=20, obstacle_density=d, seed=42)
    c = compare_ucs_and_astar(grid=rand_grid)
    print(
        f"{d * 100:>5.1f}%     | "
        f"{len(rand_grid.obstacles):<10} | "
        f"{c.ucs_result.nodes_expanded:<10} | "
        f"{c.astar_result.nodes_expanded:<10} | "
        f"{c.astar_result.path_cost:<10} | "
        f"{c.expansion_reduction_percent:>6.2f}%"
    )

## 6. Theoretical Foundations & Mathematical Justification

In [ ]:
print(get_theoretical_analysis())

## 7. University Viva Voce Reference Guide

In [ ]:
print(get_viva_reference())